# 🧊 TRELLIS 3D Generator – Gradio Interface
تحويل الصور إلى مجسمات ثلاثية الأبعاد باستخدام [TRELLIS](https://github.com/JeffreyXiang/TRELLIS).

**الميزات:**
- 📤 رفع صورة أو استعمال رابط.
- 🎛️ تحكم بالـ Seed و CFG Scale.
- ⚡ دعم Flash Attention 2 تلقائياً.
- 🖼️ عرض النموذج (GLB) وتحميله.

**التعليمات:** شغّل الخلايا **بالترتيب** من الأعلى إلى الأسفل.

In [ ]:
import torch, subprocess, sys

print(f"PyTorch: {torch.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"🔹 GPU: {gpu_name}")

SUPPORTED_GPU = ["A100", "A6000", "3090", "4090", "L4", "L40", "H100"]
USE_FLASH_ATTN = any(g in gpu_name for g in SUPPORTED_GPU)
print("✨ Flash Attention 2 مدعوم" if USE_FLASH_ATTN else "ℹ️  يستخدم SDPA (انتباه PyTorch الافتراضي).")

In [ ]:
import pkg_resources

def is_installed(pkg):
    try:
        pkg_resources.get_distribution(pkg)
        return True
    except:
        return False

print("🔍 فحص الحزم الأساسية...")
base_pkgs = ["trimesh", "pygltflib", "viser", "omegaconf", "opencv-python", "imageio", "gradio", "huggingface_hub"]
for pkg in base_pkgs:
    if not is_installed(pkg):
        !pip install -q {pkg}
        print(f"✔️ {pkg}")
    else:
        print(f"✅ {pkg}")

# spconv
if not is_installed("spconv"):
    cuda_ver = torch.version.cuda.replace(".", "")
    try:
        !pip install -q spconv-cu{cuda_ver}
    except:
        !pip install -q spconv
    print("✔️ spconv")
else:
    print("✅ spconv")

# Flash Attention 2 (اختياري)
if USE_FLASH_ATTN and not is_installed("flash-attn"):
    print("⚡ تثبيت Flash Attention 2...")
    !pip install -q ninja packaging
    !pip install flash-attn --no-build-isolation 2>&1 | tail -5
    import flash_attn
    print("✅ flash-attn")

print("\n✅ جميع الحزم الأساسية جاهزة.")

In [ ]:
import os, sys, zipfile, io, requests

ZIP_URL = "https://codeload.github.com/JeffreyXiang/TRELLIS/zip/refs/heads/main"
TARGET_DIR = "/content/TRELLIS-main"

if not os.path.isdir(TARGET_DIR):
    print("⬇️  تحميل المستودع من GitHub...")
    try:
        r = requests.get(ZIP_URL, timeout=30)
        r.raise_for_status()
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            z.extractall("/content")
        print("✅ تم تنزيل وفك ضغط المستودع.")
    except Exception as e:
        print(f"❌ فشل التحميل: {e}")
        print("يرجى رفع الملف يدوياً من https://github.com/JeffreyXiang/TRELLIS/archive/refs/heads/main.zip")
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            with zipfile.ZipFile(io.BytesIO(uploaded[fn])) as z:
                z.extractall("/content")
            print("✅ تم فك الضغط.")

if os.path.isdir(TARGET_DIR):
    sys.path.insert(0, TARGET_DIR)
    print(f"📁 تم إضافة {TARGET_DIR} إلى مسار Python.")
else:
    raise FileNotFoundError("لم يتم العثور على مجلد TRELLIS-main.")

try:
    import trellis
    print("✅ استيراد trellis ناجح.")
except Exception as e:
    print(f"❌ فشل استيراد trellis: {e}")
    raise

# إعدادات الانتباه
if USE_FLASH_ATTN:
    try:
        import flash_attn
        attn_impl = "flash_attention_2"
        print("🔧 Flash Attention 2 مفعل.")
    except:
        attn_impl = "sdpa"
        print("🔧 Flash Attention غير متاح، استخدام SDPA.")
else:
    attn_impl = "sdpa"
    print("🔧 SDPA مفعل.")

In [ ]:
import torch
from trellis.pipelines import TrellisImageTo3DPipeline

try:
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16,
        attn_implementation=attn_impl
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب.")
except Exception as e:
    print(f"⚠️ خطأ أثناء التحميل: {e}")
    print("محاولة التحميل بدون تحديد attn_implementation...")
    pipeline = TrellisImageTo3DPipeline.from_pretrained(
        "JeffreyXiang/TRELLIS-image-large",
        torch_dtype=torch.float16
    )
    pipeline.to("cuda")
    print("✅ تم تحميل خط الأنابيب بالوضع الافتراضي.")

In [ ]:
# اختبار سريع للتأكد من أن كل شيء يعمل قبل Gradio
from PIL import Image
import requests, tempfile, os

print("⏳ تشغيل اختبار سريع (صورة تجريبية)...")
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
test_image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(test_image)

try:
    outputs = pipeline.run(test_image, seed=42)
    mesh = outputs.get('mesh')
    if mesh is not None:
        tmp = tempfile.NamedTemporaryFile(suffix=".glb", delete=False)
        mesh.export(tmp.name)
        print(f"✅ اختبار ناجح! تم توليد المجسم: {tmp.name}")
    else:
        print("⚠️ لم يتم توليد شبكة.")
except Exception as e:
    print(f"❌ فشل الاختبار: {e}")
    raise

In [ ]:
import gradio as gr
from PIL import Image
import tempfile, shutil, numpy as np, os, traceback

def generate_3d(image_input, seed, cfg_scale):
    if image_input is None:
        raise gr.Error("يرجى رفع صورة.")

    if isinstance(image_input, np.ndarray):
        image = Image.fromarray(image_input).convert("RGB")
    else:
        image = image_input.convert("RGB")

    try:
        try:
            outputs = pipeline.run(image, seed=int(seed), cfg_scale=float(cfg_scale))
        except TypeError:
            outputs = pipeline.run(image, seed=int(seed))
    except Exception as e:
        traceback.print_exc()
        raise gr.Error(f"فشل التوليد: {e}")

    mesh = outputs.get('mesh')
    if mesh is None:
        raise gr.Error("لم يتم توليد شبكة (mesh).")

    tmpdir = tempfile.mkdtemp()
    glb_path = os.path.join(tmpdir, "model.glb")
    try:
        mesh.export(glb_path)
    except Exception as e:
        shutil.rmtree(tmpdir)
        raise gr.Error(f"فشل تصدير GLB: {e}")

    return glb_path, glb_path


with gr.Blocks(title="TRELLIS 3D", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # 🧊 TRELLIS – تحويل الصورة إلى مجسم ثلاثي الأبعاد
    يدعم Flash Attention 2 تلقائياً. أرفع صورة واضغط على **توليد**.
    """)

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="الصورة المدخلة", type="pil", image_mode="RGB")
            with gr.Accordion("⚙️ إعدادات متقدمة", open=False):
                seed = gr.Number(value=42, label="البذرة (Seed)", precision=0)
                cfg_scale = gr.Slider(minimum=1.0, maximum=20.0, value=7.5, step=0.5, label="CFG Scale")
            generate_btn = gr.Button("🚀 توليد", variant="primary")
        with gr.Column():
            model_output = gr.Model3D(label="النموذج الثلاثي الأبعاد")
            download_btn = gr.File(label="تحميل ملف GLB")

    generate_btn.click(
        fn=generate_3d,
        inputs=[image_input, seed, cfg_scale],
        outputs=[model_output, download_btn]
    )

print("🌐 تشغيل واجهة Gradio...")
demo.queue(max_size=3).launch(share=True, server_port=7860, show_error=True)